# Pando R tutorial: human brain organoid GLI3

This notebook follows the original `new_grn/tutorials` steps 00–09. It inspects saved intermediate data, reconstructs the network from per-target fitted models, and writes regenerated outputs to `results/generated/pando_export/`. It does **not** rerun the expensive preprocessing and all 3,846 model fits. For a selected-target refit see the end of Step 07. The historical full-run scripts are retained in `src/legacy/pando_original/` for provenance; many need path and environment adaptation.


In [ ]:
root <- normalizePath(getwd(), mustWork=TRUE)
while (!dir.exists(file.path(root, "dataset")) || !dir.exists(file.path(root, "results"))) {
  parent <- dirname(root)
  if (parent == root) stop("Start Jupyter from within the repository")
  root <- parent
}
Sys.setenv(GRNM_ROOT=root)
pando <- file.path(root, "dataset", "pando", "intermediates")
result <- file.path(root, "results", "pando")
suppressPackageStartupMessages(library(data.table))
cat("Repository:", root, "\nR:", R.version.string, "\n")


## 00 · Input identity
RNA and ATAC were measured in separate cells. Small metadata records are included; large original matrices are excluded and can be downloaded separately as described in the README.


In [ ]:
cells <- readRDS(file.path(pando, "audit", "cell_names.rds"))
cat("ATAC cells:",length(cells[[1]]),"RNA cells:",length(cells[[2]]),"\n")
stopifnot(length(cells[[1]])==38957, length(cells[[2]])==50000)
cat("Large raw RNA/ATAC objects are separately downloaded; see README.\n")


## 01 · Genomic resources
The historical analysis uses hg38 candidate regions, motif-to-TF mappings and EnsDb v86 annotations.


In [ ]:
features <- readRDS(file.path(pando,"resources","grn_features.rds"))
stopifnot(length(features)==4342)
cat("GRN features:",length(features),"\n")
stopifnot(all(file.exists(file.path(pando,"resources",c("motifs.rds","motif2tf.rds","paper_candidate_prior_regions.hg38.rds","EnsDb.Hsapiens.v86.genes.hg38.rds")))))


## 02 · Fragments and gene activity
Saved GeneActivity checkpoints derive from ATAC fragments; expression for modeling comes from measured RNA. This inspection does not recompute the matrix.


In [ ]:
activity_qc <- fread(file.path(pando,"inputs","GeneActivity_QC.csv"))
print(activity_qc[,.(batch,cells,genes,min_gene_activity)])
stopifnot(sum(activity_qc$cells)==38957, all(activity_qc$genes==19607),all(activity_qc$min_gene_activity>0))
if (all(file.exists(file.path(pando,"fragments",activity_qc$batch,"fragments.tsv.gz")))) cat("Fragment files present.\n") else cat("Large fragments are not included; see README.\n")


## 03 · CCA and cross-modality matching
Pairs are grouped by age and cell line; repeated cell identities do not represent independent samples.


In [ ]:
pairs <- fread(file.path(pando,"matches","all_pairs.tsv"))
groups <- readRDS(file.path(pando,"cca","group_summary.rds"))
cat("Matched pairs:",nrow(pairs),"groups:",length(groups),"\n")
stopifnot(nrow(pairs)==47731,length(groups)==36,all(pairs$ATAC %in% cells[[1]]),all(pairs$RNA %in% cells[[2]]))


## 04 · Reference classifier for QC
The saved rank-based multinomial LASSO uses WGCNAcluster instead of the original MainCellType label.


In [ ]:
classifier <- readRDS(file.path(pando,"qc","nowakowski_rank_lasso.rds"))
cat("Reference cells:",length(classifier$foldid),"genes:",length(classifier$genes),"label:",classifier$label_field,"\n")
stopifnot(length(classifier$foldid)==4261,length(classifier$genes)==20477,classifier$label_field=="WGCNAcluster")


## 05 · Metacells and pseudocells
RNA and ATAC are aggregated after matching. Check saved membership and accessibility cluster tables.


In [ ]:
meta <- fread(file.path(pando,"coarse","metacell_QC.csv"))
member <- fread(file.path(pando,"coarse","pseudocell_membership.tsv"))
clusters <- fread(file.path(pando,"coarse","accessibility_clusters.csv"))
cat("Metacells:",nrow(meta),"retained:",sum(!meta$glycolysis_excluded),"pseudocells:",uniqueN(member$pseudocell),"accessibility clusters:",uniqueN(clusters$accessibility_cluster),"\n")
stopifnot(nrow(meta)==27427,sum(!meta$glycolysis_excluded)==22006,uniqueN(member$pseudocell)==6612,uniqueN(clusters$accessibility_cluster)==676)


## 06 · Motif mapping
The motif threshold used in scanning is distinct from the subsequent GRN FDR.


In [ ]:
motif_file <- file.path(pando,"motifs","paper_motifs.rds")
if (file.exists(motif_file)) {
 suppressPackageStartupMessages({library(GenomicRanges);library(Matrix)})
 motif <- readRDS(motif_file)
 cat("Scanned regions:",nrow(motif$motifs),"motifs:",ncol(motif$motifs),"nonzero motif hits:",length(motif$motifs@x),"\n")
 stopifnot(nrow(motif$motifs)==245710,ncol(motif$motifs)==1122,length(motif$motifs@x)==7600744)
 rm(motif);gc()
} else {
 summary <- fread(file.path(pando,"motifs","motif_summary.csv"))
 print(summary)
 cat("Saved motif summary shown; large motif matrix not included, so it was not rechecked.\n")
}


## 07 · Per-target GLM fits
Saved models implement Gaussian GLMs using TF expression × peak accessibility interactions. A selected target can be refit separately with `GRN_FIT_TARGETS=HES5 Rscript src/pando/refit_selected_targets.R`; the full set is computationally expensive.


In [ ]:
fit_files <- list.files(file.path(pando,"fits"),pattern="^[0-9]+_.*\\.rds$",full.names=TRUE)
status <- fread(file.path(result,"target_status.csv"))
cat("Fit checkpoints:",length(fit_files),"fitted:",sum(status$status=="fit"),"\n")
stopifnot(length(fit_files)==3846,sum(status$status=="fit")==3446,!any(status$status=="error"))
example <- readRDS(fit_files[1]); print(example$qc)


## 08 · BH correction and TF–gene network
Reconstruct all coefficients from the saved target fits, apply BH across all valid coefficient tests, aggregate significant regions, to produce the TF–gene network.


In [ ]:
source(file.path(root,"src/pando/export_saved_fits.R"),local=new.env(parent=globalenv()))
original <- fread(file.path(root,"results/generated/pando_export/TF_gene_network.csv"))
cat("Exported TF–gene network:",nrow(original),"edges\n")


## 09 · Inspect GLI3 result
Coefficients describe conditional associations; they do not independently prove regulation or direct binding.


In [ ]:
gli3 <- original[tf=="GLI3"][order(padj_min,-abs(estimate),target)]
print(gli3[,.(tf,target,estimate,padj_min)][1:min(10,.N)])
cat("GLI3 edges:",nrow(gli3),"\nR_TUTORIAL_COMPLETE\n")
